### Get uniprot sequences for CPDB entries

In [1]:
import pandas as pd

fasta_dir = '/home/ubuntu/CIRPIN/cpdb_comparison/foldseek_CPDB_comparison/cpdb_pdbs_2/fasta_seqs/'
fp = "/home/ubuntu/CIRPIN/cpdb_comparison/foldseek_CPDB_comparison/cpdb_pdbs_2/CPDB_full_data_ark.csv"
df = pd.read_csv(fp)


In [2]:
import requests
import pandas as pd
import time
from typing import List, Dict, Optional

# Your PDB list (with chain identifiers like '1a0nA')
cpdb_all_names = list(set(df['Protein1']).union(df['Protein2']))

In [5]:
def parse_pdb_chain(pdb_chain: str) -> tuple:
    """
    Parse PDB ID with chain identifier.
    
    Args:
        pdb_chain: PDB with chain (e.g., '1a0nA')
    
    Returns:
        Tuple of (pdb_id, chain_id)
    """
    # Assume last character is chain ID
    if len(pdb_chain) > 4:
        pdb_id = pdb_chain[:-1].lower()
        chain_id = pdb_chain[-1]
    else:
        pdb_id = pdb_chain.lower()
        chain_id = None
    
    return pdb_id, chain_id

def get_uniprot_for_chain(pdb_id: str, chain_id: str) -> Optional[List[str]]:
    """
    Fetch UniProt IDs for a specific chain of a PDB structure.
    
    Args:
        pdb_id: PDB identifier (e.g., '1a0n')
        chain_id: Chain identifier (e.g., 'A')
    
    Returns:
        List of UniProt IDs for the specified chain
    """
    url = f"https://www.ebi.ac.uk/pdbe/api/mappings/uniprot/{pdb_id}"
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        uniprot_ids = []
        
        # Navigate through the data structure
        if pdb_id in data and "UniProt" in data[pdb_id]:
            for uniprot_id, uniprot_data in data[pdb_id]["UniProt"].items():
                # Check mappings for this chain
                if "mappings" in uniprot_data:
                    for mapping in uniprot_data["mappings"]:
                        if mapping.get("chain_id") == chain_id:
                            uniprot_ids.append(uniprot_id)
                            break  # Found this UniProt for this chain
        
        return uniprot_ids if uniprot_ids else None
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {pdb_id}: {e}")
        return None
    except Exception as e:
        print(f"Unexpected error for {pdb_id}: {e}")
        return None

def map_pdb_chain_to_uniprot(pdb_chain_list: List[str], delay: float = 0.1) -> pd.DataFrame:
    """
    Map PDB IDs with chains to UniProt IDs.
    
    Args:
        pdb_chain_list: List of PDB IDs with chains (e.g., ['1a0nA', '2bcdB'])
        delay: Delay between API calls in seconds
    
    Returns:
        DataFrame with PDB_Chain, PDB_ID, Chain_ID, and UniProt_ID columns
    """
    results = []
    processed_pdbs = set()
    
    for i, pdb_chain in enumerate(pdb_chain_list):
        pdb_id, chain_id = parse_pdb_chain(pdb_chain)
        
        print(f"Processing {i+1}/{len(pdb_chain_list)}: {pdb_chain} (PDB: {pdb_id}, Chain: {chain_id})")
        
        # Only query API once per unique PDB (not per chain)
        if pdb_id not in processed_pdbs:
            time.sleep(delay)
            processed_pdbs.add(pdb_id)
        
        if chain_id:
            uniprot_ids = get_uniprot_for_chain(pdb_id, chain_id)
            
            if uniprot_ids:
                # Create a row for each UniProt ID
                for uniprot_id in uniprot_ids:
                    results.append({
                        'PDB_Chain': pdb_chain,
                        'PDB_ID': pdb_id,
                        'Chain_ID': chain_id,
                        'UniProt_ID': uniprot_id
                    })
            else:
                # Keep entries with no mapping
                results.append({
                    'PDB_Chain': pdb_chain,
                    'PDB_ID': pdb_id,
                    'Chain_ID': chain_id,
                    'UniProt_ID': None
                })
        else:
            # No chain specified
            results.append({
                'PDB_Chain': pdb_chain,
                'PDB_ID': pdb_id,
                'Chain_ID': None,
                'UniProt_ID': None
            })
    
    return pd.DataFrame(results)

# Run the mapping
print(f"Mapping {len(cpdb_all_names)} PDB+Chain IDs to UniProt...")
mapping_df = map_pdb_chain_to_uniprot(cpdb_all_names)

# Save to CSV
output_file = 'pdb_to_uniprot_mapping_chain.csv'
mapping_df.to_csv(output_file, index=False)

print(f"\nMapping complete!")
print(f"Saved to: {output_file}")
print(f"\nSummary:")
print(f"Total PDB+Chain entries: {len(cpdb_all_names)}")
print(f"Total mappings: {len(mapping_df)}")
print(f"Entries with UniProt mappings: {mapping_df['UniProt_ID'].notna().sum()}")
print(f"Entries without mappings: {mapping_df['UniProt_ID'].isna().sum()}")

# Show first few rows
print("\nFirst few mappings:")
print(mapping_df.head(10))

# Show chain distribution
print("\nChain distribution:")
print(mapping_df['Chain_ID'].value_counts().head(10))

Mapping 2238 PDB+Chain IDs to UniProt...
Processing 1/2238: 1orjA (PDB: 1orj, Chain: A)
Processing 2/2238: 1w8gA (PDB: 1w8g, Chain: A)
Processing 3/2238: 1jpjA (PDB: 1jpj, Chain: A)
Processing 4/2238: 1iruF (PDB: 1iru, Chain: F)
Processing 5/2238: 2h7vA (PDB: 2h7v, Chain: A)
Processing 6/2238: 2bbkH (PDB: 2bbk, Chain: H)
Processing 7/2238: 2ghsA (PDB: 2ghs, Chain: A)
Processing 8/2238: 1uv7A (PDB: 1uv7, Chain: A)
Processing 9/2238: 1jq0A (PDB: 1jq0, Chain: A)
Processing 10/2238: 2fb0A (PDB: 2fb0, Chain: A)
Processing 11/2238: 2d1zA (PDB: 2d1z, Chain: A)
Processing 12/2238: 1nccN (PDB: 1ncc, Chain: N)
Processing 13/2238: 1j2qH (PDB: 1j2q, Chain: H)
Processing 14/2238: 2imgA (PDB: 2img, Chain: A)
Processing 15/2238: 1tqjA (PDB: 1tqj, Chain: A)
Processing 16/2238: 1gncA (PDB: 1gnc, Chain: A)
Processing 17/2238: 1c5hA (PDB: 1c5h, Chain: A)
Processing 18/2238: 1wifA (PDB: 1wif, Chain: A)
Processing 19/2238: 2h16A (PDB: 2h16, Chain: A)
Processing 20/2238: 1f70A (PDB: 1f70, Chain: A)
Processi

### Note that after finding the uniprot associated with each pdb chain, there are some that have two uniprot IDs
### These represent cases of fusion proteins or chimeric proteins such as cat allergen (1puo, 1zkr).


In [25]:
num_fusions = len(mapping_df['PDB_Chain'].value_counts()[lambda x: x > 1])

In [45]:
len(mapping_df)

2248

In [33]:
ex = mapping_df[mapping_df['PDB_Chain'] == '1puoA']

In [41]:
mapping_df['PDB_Chain'].value_counts()[lambda x: x > 1]

PDB_Chain
1sy6A    2
1zkrA    2
1tmhA    2
1puoA    2
1mowA    2
1jbjA    2
1e9fA    2
1j2oA    2
1xmwA    2
1axkA    2
Name: count, dtype: int64

In [34]:
print(f'Fusion/chimeric proteins: {num_fusions}')
print(f'Example: {ex}')


Fusion/chimeric proteins: 10
Example:      PDB_Chain PDB_ID Chain_ID UniProt_ID
2018     1puoA   1puo        A     P30440
2019     1puoA   1puo        A     P30438


In [35]:
mapping_df.head()

,PDB_Chain,PDB_ID,Chain_ID,UniProt_ID
0,1orjA,1orj,A,O67806
1,1w8gA,1w8g,A,P67080
2,1jpjA,1jpj,A,O07347
3,1iruF,1iru,F,Q3T0X5
4,2h7vA,2h7v,A,P63000


In [36]:

def get_uniprot_sequence(uniprot_id: str) -> Optional[Dict[str, str]]:
    """
    Fetch protein sequence for a given UniProt ID.
    
    Args:
        uniprot_id: UniProt accession (e.g., 'P07751')
    
    Returns:
        Dictionary with sequence info or None if failed
    """
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        # Parse FASTA format
        fasta_text = response.text
        lines = fasta_text.strip().split('\n')
        
        if len(lines) < 2:
            return None
        
        # First line is header (starts with >)
        header = lines[0]
        # Remaining lines are sequence
        sequence = ''.join(lines[1:])
        
        return {
            'uniprot_id': uniprot_id,
            'header': header,
            'sequence': sequence,
            'length': len(sequence)
        }
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {uniprot_id}: {e}")
        return None
    except Exception as e:
        print(f"Unexpected error for {uniprot_id}: {e}")
        return None

def fetch_sequences_for_mapping(mapping_df: pd.DataFrame, delay: float = 0.1) -> pd.DataFrame:
    """
    Fetch sequences for all UniProt IDs in the mapping DataFrame.
    
    Args:
        mapping_df: DataFrame with UniProt_ID column
        delay: Delay between API calls in seconds
    
    Returns:
        DataFrame with sequences added
    """
    # Get unique UniProt IDs (excluding None)
    unique_uniprot = mapping_df['UniProt_ID'].dropna().unique()
    
    # Fetch sequences
    sequences = {}
    print(f"Fetching sequences for {len(unique_uniprot)} unique UniProt IDs...")
    
    for i, uniprot_id in enumerate(unique_uniprot):
        print(f"Processing {i+1}/{len(unique_uniprot)}: {uniprot_id}")
        
        seq_info = get_uniprot_sequence(uniprot_id)
        if seq_info:
            sequences[uniprot_id] = seq_info
        if i % 100 == 0:
            print(seq_info)
        time.sleep(delay)
    
    # Add sequences to mapping DataFrame
    mapping_df['Sequence'] = mapping_df['UniProt_ID'].map(
        lambda x: sequences[x]['sequence'] if x in sequences else None
    )
    mapping_df['Sequence_Length'] = mapping_df['UniProt_ID'].map(
        lambda x: sequences[x]['length'] if x in sequences else None
    )
    mapping_df['Header'] = mapping_df['UniProt_ID'].map(
        lambda x: sequences[x]['header'] if x in sequences else None
    )
    
    return mapping_df

# Fetch sequences
mapping_with_seq = fetch_sequences_for_mapping(mapping_df)

# Save to CSV
output_file = 'pdb_uniprot_with_sequences.csv'
mapping_with_seq.to_csv(output_file, index=False)

print(f"\nSequences fetched and saved to: {output_file}")
print(f"\nSummary:")
print(f"Total mappings: {len(mapping_with_seq)}")
print(f"Sequences retrieved: {mapping_with_seq['Sequence'].notna().sum()}")
print(f"Failed retrievals: {mapping_with_seq['Sequence'].isna().sum()}")

# Show first few rows (truncate sequence for display)
display_df = mapping_with_seq.copy()
display_df['Sequence_Preview'] = display_df['Sequence'].apply(
    lambda x: x[:50] + '...' if pd.notna(x) and len(x) > 50 else x
)
print("\nFirst few entries:")
print(display_df[['PDB_ID', 'UniProt_ID', 'Sequence_Length', 'Sequence_Preview']].head(10))

# Optional: Save as FASTA file
def save_as_fasta(df: pd.DataFrame, output_file: str):
    """Save sequences in FASTA format."""
    with open(output_file, 'w') as f:
        for _, row in df.iterrows():
            if pd.notna(row['Sequence']):
                # Use the original header or create one
                if pd.notna(row['Header']):
                    f.write(f"{row['Header']}\n")
                else:
                    f.write(f">{row['UniProt_ID']} | PDB:{row['PDB_ID']}\n")
                f.write(f"{row['Sequence']}\n")

# Uncomment to save as FASTA
# save_as_fasta(mapping_with_seq, 'sequences.fasta')
# print("FASTA file saved to: sequences.fasta")

Fetching sequences for 1443 unique UniProt IDs...
Processing 1/1443: O67806
{'uniprot_id': 'O67806', 'header': '>tr|O67806|O67806_AQUAE Flagellar protein FliS OS=Aquifex aeolicus (strain VF5) OX=224324 GN=fliS PE=1 SV=1', 'sequence': 'MRNIAEAYFQNMVETATPLEQIILLYDKAIECLERAIEIYDQVNELEKRKEFVENIDRVYDIISALKSFLDHEKGKEIAKNLDTIYTIILNTLVKVDKTKEELQKILEILKDLREAWEEVKKKV', 'length': 124}
Processing 2/1443: P67080
Processing 3/1443: O07347
Processing 4/1443: Q3T0X5
Processing 5/1443: P63000
Processing 6/1443: P29894
Processing 7/1443: Q7D0W3
Processing 8/1443: P41851
Processing 9/1443: S4WCF9
Processing 10/1443: Q8A7U6
Processing 11/1443: Q7SI98
Processing 12/1443: P03472
Processing 13/1443: Q9P996
Processing 14/1443: Q9BVJ7
Processing 15/1443: P74061
Processing 16/1443: P09919
Processing 17/1443: P09850
Processing 18/1443: Q9D9M4
Processing 19/1443: Q9Y689
Processing 20/1443: P0DP33
Processing 21/1443: P0A7E1
Processing 22/1443: P06992
Processing 23/1443: Q8N4E7
Processing 24/1443: P0A434
Processing

In [37]:
mapping_with_seq.head()

,PDB_Chain,PDB_ID,Chain_ID,UniProt_ID,Sequence,Sequence_Length,Header
0,1orjA,1orj,A,O67806,MRNIAEAYFQNMVETATPLEQIILLYDKAIECLERAIEIYDQVNEL...,124.0,>tr|O67806|O67806_AQUAE Flagellar protein FliS...
1,1w8gA,1w8g,A,P67080,MNDIAHNLAQVRDKISAAATRCGRSPEEITLLAVSKTKPASAIAEA...,234.0,>sp|P67080|PLPHP_ECOLI Pyridoxal phosphate hom...
2,1jpjA,1jpj,A,O07347,MFQQLSARLQEAIGRLRGRGRITEEDLKATLREIRRALMDADVNLE...,430.0,>sp|O07347|SRP54_THEAQ Signal recognition part...
3,1iruF,1iru,F,Q3T0X5,MFRNQYDNDVTVWSPQGRIHQIEYAMEAVKQGSATVGLKSKTHAVL...,263.0,>sp|Q3T0X5|PSA1_BOVIN Proteasome subunit alpha...
4,2h7vA,2h7v,A,P63000,MQAIKCVVVGDGAVGKTCLLISYTTNAFPGEYIPTVFDNYSANVMV...,192.0,>sp|P63000|RAC1_HUMAN Ras-related C3 botulinum...


In [38]:
import os
from pathlib import Path

def read_fasta_sequence(fasta_path):
    """Read sequence from a FASTA file, ignoring header lines."""
    sequence = []
    with open(fasta_path, 'r') as f:
        for line in f:
            if not line.startswith('>'):
                sequence.append(line.strip())
    return ''.join(sequence)

def get_pdb_sequence(pdb_chain, fasta_dir):
    """
    Get PDB sequence for a given PDB chain.
    Converts format from '1fl9A' to '1FL9_A.fasta'
    """
    if pd.isna(pdb_chain) or pdb_chain == '':
        return None
    
    # Split PDB ID and chain
    # Assuming format is like '1fl9A' where last character is chain
    pdb_id = pdb_chain[:-1].upper()
    chain = pdb_chain[-1].upper()
    
    # Construct filename
    fasta_filename = f"{pdb_id}_{chain}.fasta"
    fasta_path = os.path.join(fasta_dir, fasta_filename)
    
    # Read sequence if file exists
    if os.path.exists(fasta_path):
        try:
            return read_fasta_sequence(fasta_path)
        except Exception as e:
            print(f"Error reading {fasta_filename}: {e}")
            return None
    else:
        print(f"File not found: {fasta_filename}")
        return None

# Set the directory containing FASTA files
fasta_dir = "/home/ubuntu/CIRPIN/cpdb_comparison/foldseek_CPDB_comparison/cpdb_pdbs_2/fasta_seqs"

# Create the new column
mapping_with_seq['pdb_sequence'] = mapping_with_seq['PDB_Chain'].apply(
    lambda x: get_pdb_sequence(x, fasta_dir)
)

# Check results
print(f"Total entries: {len(mapping_with_seq)}")
print(f"Sequences found: {mapping_with_seq['pdb_sequence'].notna().sum()}")
print(f"Sequences missing: {mapping_with_seq['pdb_sequence'].isna().sum()}")

# Display a few examples
print("\nSample entries:")
print(mapping_with_seq[['PDB_Chain', 'pdb_sequence']].head())

Total entries: 2248
Sequences found: 2248
Sequences missing: 0

Sample entries:
  PDB_Chain                                       pdb_sequence
0     1orjA  RNIAEAYFQNMVETATPLEQIILLYDKAIECLERAIEIYDQVNELE...
1     1w8gA  DIAHNLAQVRDKISAAATRCGRSPEEITLLAVSKTKPASAIAEAID...
2     1jpjA  MFQQLSARLQEAIGRLRGRGRITEEDLKATLREIRRALMDADVNLE...
3     1iruF  NQYDNDVTVWSPQGRIHQIEYAMEAVKQGSATVGLKSKTHAVLVAL...
4     2h7vA                                                 GS


In [39]:
mapping_with_seq

,PDB_Chain,PDB_ID,Chain_ID,UniProt_ID,Sequence,Sequence_Length,Header,pdb_sequence
0,1orjA,1orj,A,O67806,MRNIAEAYFQNMVETATPLEQIILLYDKAIECLERAIEIYDQVNEL...,124.0,>tr|O67806|O67806_AQUAE Flagellar protein FliS...,RNIAEAYFQNMVETATPLEQIILLYDKAIECLERAIEIYDQVNELE...
1,1w8gA,1w8g,A,P67080,MNDIAHNLAQVRDKISAAATRCGRSPEEITLLAVSKTKPASAIAEA...,234.0,>sp|P67080|PLPHP_ECOLI Pyridoxal phosphate hom...,DIAHNLAQVRDKISAAATRCGRSPEEITLLAVSKTKPASAIAEAID...
2,1jpjA,1jpj,A,O07347,MFQQLSARLQEAIGRLRGRGRITEEDLKATLREIRRALMDADVNLE...,430.0,>sp|O07347|SRP54_THEAQ Signal recognition part...,MFQQLSARLQEAIGRLRGRGRITEEDLKATLREIRRALMDADVNLE...
3,1iruF,1iru,F,Q3T0X5,MFRNQYDNDVTVWSPQGRIHQIEYAMEAVKQGSATVGLKSKTHAVL...,263.0,>sp|Q3T0X5|PSA1_BOVIN Proteasome subunit alpha...,NQYDNDVTVWSPQGRIHQIEYAMEAVKQGSATVGLKSKTHAVLVAL...
4,2h7vA,2h7v,A,P63000,MQAIKCVVVGDGAVGKTCLLISYTTNAFPGEYIPTVFDNYSANVMV...,192.0,>sp|P63000|RAC1_HUMAN Ras-related C3 botulinum...,GS
...,...,...,...,...,...,...,...,...
2243,1hl2A,1hl2,A,P0A6L4,MATNLRGVMAALLTPFDQQQALDKASLRRLVQFNIQQGIDGLYVGG...,297.0,>sp|P0A6L4|NANA_ECOLI N-acetylneuraminate lyas...,TNLRGVMAALLTPFDQQQALDKASLRRLVQFNIQQGIDGLYVGGST...
2244,1dazC,1daz,C,P03367,MGARASVLSGGELDRWEKIRLRPGGKKKYKLKHIVWASRELERFAV...,1447.0,>sp|P03367|POL_HV1BR Gag-Pol polyprotein OS=Hu...,PQITLWKRPLVTIKIGGQLKEALLDTGADDTVIEEMSLPGRWKPIM...
2245,2h8xA,2h8x,A,Q88NF7,MSALFEPYTLKDVTLRNRIAIPPMCQYMAEDGMINDWHHVHLAGLA...,363.0,>tr|Q88NF7|Q88NF7_PSEPK XenA OS=Pseudomonas pu...,ALFEPYTLKDVTLRNRIAIPPMCQYMAEDGMINDWHHVHLAGLARG...
2246,1n5bA,1n5b,A,P31490,MYSFEQAITQLFQQLSLSIPDTIEPVIGVKVGEFACHITEHPVGQI...,130.0,>sp|P31490|YERA_YERE8 YopE regulator OS=Yersin...,SFEQAITQLFQQLSLSIPDTIEPVIGVKVGEFACHITEHPVGQILM...


In [40]:
output_file = 'pdb_uniprot_with_uniprot_and_fasta.csv'
mapping_with_seq.to_csv(output_file, index=False)

In [ ]:
####### Same uniprot IDs